In [5]:
# Updated import path
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [7]:
import pandas as pd

# Using the absolute path to your file
file_path = r"C:\Users\gabyt\Documents\Data Science\Labs\Module 3\lab-natural-language-processing\data\kg_train.csv"
data = pd.read_csv(file_path, encoding='latin-1')

# Reduce the training set to speed up development
data = data.head(1000)
print(data.shape)
data.fillna("", inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [8]:
from sklearn.model_selection import train_test_split

# Split the data into 80% training and 20% validation
data_train, data_val = train_test_split(data, test_size=0.2, random_state=42)

print(f"Train set size: {data_train.shape}")
print(f"Validation set size: {data_val.shape}")

Train set size: (800, 2)
Validation set size: (200, 2)


## Data Preprocessing

In [ ]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [9]:
import re

def clean_html(text):
    text = str(text)
    # 1. Remove inline JavaScript/CSS (Notice the DOTALL flag so it catches newlines)
    text = re.sub(r'<script.*?</script>|<style.*?</style>', '', text, flags=re.DOTALL|re.IGNORECASE)
    # 2. Remove html comments
    text = re.sub(r'', '', text, flags=re.DOTALL)
    # 3. Remove remaining HTML tags
    text = re.sub(r'<.*?>', '', text)
    return text

# We use .copy() to prevent pandas SettingWithCopy warnings
data_train = data_train.copy()
data_val = data_val.copy()

data_train['cleaned_text'] = data_train['text'].apply(clean_html)
data_val['cleaned_text'] = data_val['text'].apply(clean_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [10]:
def clean_text(text):
    # Remove all the special characters
    text = re.sub(r'\W', ' ', text)
    # Remove numbers
    text = re.sub(r'\d+', ' ', text)
    # Remove all single characters bounded by spaces
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    # Remove single characters from the start
    text = re.sub(r'^[a-zA-Z]\s+', ' ', text)
    # Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    # Remove prefixed 'b' (often an artifact of byte strings)
    text = re.sub(r'^b\s+', '', text)
    # Convert to Lowercase
    text = text.lower().strip()
    return text

data_train['preprocessed_text'] = data_train['cleaned_text'].apply(clean_text)
data_val['preprocessed_text'] = data_val['cleaned_text'].apply(clean_text)

## Now let's work on removing stopwords
Remove the stopwords.

In [12]:
import nltk

# Download the required NLTK data files to your local machine
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gabyt\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gabyt\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\gabyt\AppData\Roaming\nltk_data...


True

In [13]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    # Split the text into words, keep it if it's not a stopword, and glue it back together
    return " ".join([word for word in text.split() if word not in stop_words])

data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(remove_stopwords)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(remove_stopwords)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [14]:
from nltk.stem import WordNetLemmatizer
import nltk

# Ensure the dictionary is downloaded
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(lemmatize_text)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(lemmatize_text)

# Let's see the clean data!
display(data_train[['label', 'preprocessed_text']].head())

,label,preprocessed_text
29,1,regard mr nelson smith kindly reply private em...
535,0,able reach oscar supposed send pdb receive
695,0,huma abedin checking pat work jack jake rest a...
557,0,announced monday today
836,1,bank africaagence san pedro bp san pedro cote ...


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_n_words(corpus, n=10):
    vec = CountVectorizer().fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    # Sort the words by frequency
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

# Filter our data into ham (0) and spam (1)
ham_text = data_train[data_train['label'] == 0]['preprocessed_text']
spam_text = data_train[data_train['label'] == 1]['preprocessed_text']

print("Top 10 HAM words:", get_top_n_words(ham_text))
print("Top 10 SPAM words:", get_top_n_words(spam_text))

Top 10 HAM words: [('state', np.int64(117)), ('pm', np.int64(97)), ('would', np.int64(94)), ('president', np.int64(89)), ('mr', np.int64(89)), ('time', np.int64(81)), ('percent', np.int64(80)), ('obama', np.int64(77)), ('call', np.int64(74)), ('secretary', np.int64(74))]
Top 10 SPAM words: [('money', np.int64(842)), ('account', np.int64(740)), ('bank', np.int64(645)), ('fund', np.int64(625)), ('transaction', np.int64(466)), ('business', np.int64(424)), ('mr', np.int64(421)), ('country', np.int64(419)), ('million', np.int64(369)), ('company', np.int64(363))]


## Extra features

In [16]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,cleaned_text,preprocessed_text,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",regard mr nelson smith kindly reply private em...,0,0,79
535,I have not been able to reach oscar this am. W...,0,I have not been able to reach oscar this am. W...,able reach oscar supposed send pdb receive,0,0,42
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,; Huma Abedin B6I'm checking with Pat on the 5...,huma abedin checking pat work jack jake rest a...,0,0,76
557,I can have it announced here on Monday - can't...,0,I can have it announced here on Monday - can't...,announced monday today,0,0,22
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,bank africaagence san pedro bp san pedro cote ...,1,1,1104


## How would you create a Bag of Words with the CountVectorizer method?

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Create Bag of Words
bow_vec = CountVectorizer()
X_train_bow = bow_vec.fit_transform(data_train['preprocessed_text'])
X_val_bow = bow_vec.transform(data_val['preprocessed_text'])
print(f"Bag of Words Shape: {X_train_bow.shape}")

# 2. Create TF-IDF
tfidf_vec = TfidfVectorizer()
X_train_tfidf = tfidf_vec.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf_vec.transform(data_val['preprocessed_text'])
print(f"TF-IDF Shape: {X_train_tfidf.shape}")

Bag of Words Shape: (800, 28297)
TF-IDF Shape: (800, 28297)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [18]:
# 2. Create TF-IDF
tfidf_vec = TfidfVectorizer()
X_train_tfidf = tfidf_vec.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf_vec.transform(data_val['preprocessed_text'])
print(f"TF-IDF Shape: {X_train_tfidf.shape}")

TF-IDF Shape: (800, 28297)


### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code